In [22]:
# imports

import os
import random
from google import genai
from openai import OpenAI
from google.genai import types
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display


In [23]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("AQ."):
    print("An API key was found, but it doesn't start AQ. please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


# Gemma4 API Calls

In [24]:
# Initialize the native Google client
# It automatically picks up the GEMINI_API_KEY environment variable
client = genai.Client()

In [3]:
def call_gemma(message):
    # 1. User message
    messages = [
       types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]    
        )
    ]

    # 2. Put system rules
    config = types.GenerateContentConfig(
        system_instruction="You are 'C' part of 3 young friends in a conversation, your personality is shy. Answer in under 60 characters. Do not explain. Be brief.",
    )
    # 3. Call the model using the native client
    response = client.models.generate_content(
        model='gemma-4-31b-it', # Swap out with your preferred Gemma 4 variant
        contents=message,
        config=config
    )

    return response.text


# Ollama Calls

In [4]:
ollama_url = "http://localhost:11434/v1"

ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [5]:
def call_ollama(person, mensaje):
    if person == "A":
        MODEL = "llama3.2"
    elif person == "B":
        MODEL = "deepseek-r1:1.5b"
    ol1_response = ollama.chat.completions.create(
        model = MODEL,
        messages = other_messages(person, mensaje)
    )
    return ol1_response.choices[0].message.content

# 3 men conversation - There is not errors exception checking
it can add other persons for example `D` guy 

In [6]:
def get_dialog(person, dialog_num, turn):
    """Simulate a dialog line for a person."""
    messages = {
        "A": [
            "Hey everyone, what are we doing this weekend?",
            "I was thinking we could go hiking.",
            "The weather forecast looks great for Saturday.",
            "Should we pack lunch or eat out?",
            "Let's meet at the trailhead at 9am then!",
        ],
        "B": [
            "Welcome! I'm so glad you're here, guys.",
            "Hiking sounds awesome, I know a great trail.",
            "Perfect, I'll bring my camera then.",
            "Let's pack lunch, it's cheaper and more fun.",
            "Sounds like a plan, I'll bring extra water!",
        ],
        "C": [
            "Can you make sandwiches?",
            "Oh I love hiking, which trail are you thinking?",
            "Great, I'll charge my phone for photos.",
            "I can make sandwiches for everyone if you want.",
            "Can't wait, this is going to be so fun!",
        ],
        "D": [
            "Hi there",
            "Count me in! ",
            "Oh I love hiking",
            "Great",
            "Can't wait!",
        ],
    }
    # Use dialog_num and turn to pick a somewhat varied line
    idx = (dialog_num + turn) % len(messages[person])
    f_dialog = f"{person}: {messages[person][idx]}"
    return [f_dialog[:1], f_dialog[3:]]

def store_line(person, line):
    """Store a dialog line in the appropriate list."""
    if person == "A":
        listA.append(line)
    elif person == "B":
        listB.append(line)
    elif person == "C":
        listC.append(line)
    # elif person == "D":
    #     listC.append(line)

In [12]:
def first_dialog():
    """Get ramdomly first message from a person"""
    first_speaker = random.choice(persons)  # Random starter for dialog 1
    starting_dialog = get_dialog(first_speaker, 0, 0)
    return starting_dialog


def gemma_messages(message):
    messages = [
       types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]    
        )
    ]
    return messages


def other_messages(person, message):
    """Ollama calls"""
    if person == "A":
        personality = "You are 'A', part of 3 young friends in a conversation, your personality is inpacient."
    elif person == "B":
        personality = "You are 'B', part of 3 young friends in a conversation, your personality is bored."
    messages = [
    {"role": "system", 
    "content": f"{personality} You have a hard budget of 60 characters total. Output ONLY the clean answer string. Do not think. Do not explain."}, 
    {"role": "user", "content": message}
    ]
    return messages

In [20]:
def get_responses(person, message):
    """Get responses from Person"""
    if person == "A" or person == "B":
        result = call_ollama(person, f"f_dialog")
    elif person == "C":
        result = call_gemma(f_dialog)
    return result

def run_llm_conversation(num_dialogs=5):
    """
    Run a conversation of `num_dialogs` rounds.

    Each dialog round has 3 turns (one per person):
      - Dialog 1: random starter, then the other two in random order.
      - Dialog N (N > 1): the FIRST RESPONDER (2nd speaker) of the previous
        dialog becomes the new starter, then the remaining two in random order.
    """
    first_speaker, f_dialog = first_dialog()  # Random starter for dialog 1

    print("=" * 50)
    print("        THREE-PERSON CONVERSATION")
    print("=" * 50)

    for dialog_num in range(num_dialogs):
        # Build the ordered speaker list for this dialog
        others = [p for p in persons if p != first_speaker]
        random.shuffle(others)
        order = [first_speaker] + others  # first speaker is fixed; others shuffled

        print(f"\n--- Dialog {dialog_num + 1} ---")
        print(f"  Turn order: {' → '.join(order)}")

        for turn, speaker in enumerate(order):
            if dialog_num == 0 and turn == 0:
                line = {speaker: f_dialog}
            else:
                if speaker == "A" or speaker == "B":
                    if speaker == "A":
                      if len(listA) != 0:
                          f_dialog = listA[-1]['A']
                    elif speaker == "B":
                      if len(listB) != 0:
                          f_dialog = listB[-1]['B']
                    result = call_ollama(speaker, f_dialog)
                    line = {speaker: result.replace('\n','')}
                elif speaker == "C":
                    if len(listC) != 0:
                        f_dialog = listC[-1]['C']
                    result = call_gemma(f_dialog)
                    line = {speaker: result}
            store_line(speaker, line)
            print(f"  {line}")

        # Rule 3: next dialog's starter is the FIRST RESPONDER (2nd speaker) of this dialog
        first_speaker = order[1]

    print("\n" + "=" * 50)

In [ ]:
# Storage lists for each person's dialogs
listA = []
listB = []
listC = []

persons = ["A", "B", "C"] # , "D"]

run_llm_conversation(5)